# GSP – Sequential Pattern Mining

Notebook điều phối các bước: khám phá dữ liệu, tiền xử lý, minh họa, chạy GSP và thực nghiệm.

## 1. Import thư viện và module

In [96]:
from pathlib import Path
import pandas as pd

import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import importlib
import data_processing
importlib.reload(data_processing)

from data_processing import DataLoader, DataPreprocessor, SequenceDatabaseManager,SequenceDatabase
from gsp import GSP

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" /"e-shop clothing 2008.csv"

## 2. Data Loading

In [14]:
raw_records = pd.read_csv(DATA_PATH, sep=";")
pd.DataFrame(raw_records).head()

,year,month,day,order,country,session ID,page 1 (main category),page 2 (clothing model),colour,location,model photography,price,price 2,page
0,2008,4,1,1,29,1,1,A13,1,5,1,28,2,1
1,2008,4,1,2,29,1,1,A16,1,6,1,33,2,1
2,2008,4,1,3,29,1,2,B4,10,2,1,52,1,1
3,2008,4,1,4,29,1,2,B17,6,6,2,38,2,1
4,2008,4,1,5,29,1,2,B8,4,3,2,52,1,1


## 3. Data Preprocessing


In [68]:

class EshopPreprocessor:

    CATEGORY = {
        1: "trousers",
        2: "skirts",
        3: "blouses",
        4: "sale"
    }

    PREFIX2CAT = {
        "A": 1,
        "B": 2,
        "C": 3,
        "P": 4
    }

    def __init__(self, raw_data):

        self.raw = raw_data.copy()

        self.df = None
        self.checks = {}

    # 1. Kiểm tra chất lượng dữ liệu

    def check_quality(self):

        print("===== DATA QUALITY =====")

        print("Shape:", self.raw.shape)

        print("\nData types:")
        print(self.raw.dtypes)

        print("\nMissing values:")
        print(self.raw.isna().sum())

        print(
            "\nDuplicate rows:",
            self.raw.duplicated().sum()
        )

        return self.raw

    # 2. Kiểm tra nghiệp vụ

    def check_business_logic(self):

        df = self.raw
        g = df.groupby("session ID")


        # Order có liên tục từ 1 đến n?
        def order_ok(s):

            values = np.sort(s.dropna().to_numpy())

            return np.array_equal(
                values,
                np.arange(1, len(s) + 1)
            )

        self.checks["session có order KHÔNG liên tục"] = int(
            (~g["order"].apply(order_ok)).sum()
        )

        # File đã được sắp xếp?
        self.checks["file đã sort theo session ID, order"] = bool(
            df.sort_values(
                ["session ID", "order"]
            ).index.equals(df.index)
        )
        print("\n=== BUSINESS LOGIC CHECK ===")

        for check, result in self.checks.items():
            print(f"{check}: {result}")

    # 3. Pipeline tiền xử lý

    def preprocess(self):

        self.check_quality()

        self.check_business_logic()

        return self.df


In [69]:
preprocessor = EshopPreprocessor(raw_records)

# Bước 1: Kiểm tra chất lượng
preprocessor.check_quality()

# Bước 2: Kiểm tra tính nhất quán
preprocessor.check_business_logic()

===== DATA QUALITY =====
Shape: (165474, 14)

Data types:
year                       int64
month                      int64
day                        int64
order                      int64
country                    int64
session ID                 int64
page 1 (main category)     int64
page 2 (clothing model)      str
colour                     int64
location                   int64
model photography          int64
price                      int64
price 2                    int64
page                       int64
dtype: object

Missing values:
year                       0
month                      0
day                        0
order                      0
country                    0
session ID                 0
page 1 (main category)     0
page 2 (clothing model)    0
colour                     0
location                   0
model photography          0
price                      0
price 2                    0
page                       0
dtype: int64

Duplicate rows: 0

=== BUSINE

## 4. EDA (nếu có)
Khám phá số lượng sequence, transaction và item trước khi khai phá.

In [113]:
rows = raw_records[
    ["session ID", "order", "page 2 (clothing model)"]
].itertuples(index=False, name=None)

records = (
    {
        "sequence_id": str(session_id),
        "transaction_id": str(order),
        "items": str(product)
    }
    for session_id, order, product in rows
)

In [114]:
manager = SequenceDatabaseManager()

database = manager.build(records)

print("Số sequence:", len(database))
print("Ví dụ:", database.sequences["1"])

manager.save("outputs/sequence_database.csv")

Số sequence: 24026
Ví dụ: (frozenset({'A13'}), frozenset({'A16'}), frozenset({'B4'}), frozenset({'B17'}), frozenset({'B8'}), frozenset({'C56'}), frozenset({'C57'}), frozenset({'P67'}), frozenset({'P82'}))
Sequence database saved to: outputs\sequence_database.csv


In [105]:
manager.save("outputs/sequence_database.csv")

Sequence database saved to: outputs\sequence_database.csv


## 5. GSP Demonstration
Dùng một sequence database nhỏ để diễn giải từng vòng lặp của thuật toán.

In [ ]:
# TODO: trình bày candidate generation, pruning và support counting từng level.

## 6. GSP Implementation

## 7. Experiments
Thử nhiều giá trị MinSup và/hoặc nhiều quy mô dữ liệu, ghi nhận số pattern và runtime.

In [ ]:
# TODO: gọi ExperimentRunner và lưu kết quả vào thư mục results/.

## 8. Library Comparison (Optional)

In [ ]:
#TODO: so sánh tập pattern, support và thời gian chạy.